# Step 5: Deploy Gateway & Interceptor

Deploy the AgentCore Gateway and Interceptor Lambda.

## Prerequisites

- ✅ Run `04-deploy-mcp-server.ipynb` first
- ✅ MCP Server Runtime ARN saved to SSM

## What This Notebook Does

1. Deploys Gateway Interceptor Lambda
2. Creates AgentCore Gateway
3. Configures OAuth token validation
4. Saves Gateway ARN to SSM

## Next Notebook

- **06-deploy-agent.ipynb**

In [ ]:
import boto3
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from config import config

lambda_client = boto3.client('lambda')
ssm_client = boto3.client('ssm')
sts_client = boto3.client('sts')

AWS_ACCOUNT_ID = sts_client.get_caller_identity()['Account']
AWS_REGION = config.AWS_REGION

print('✅ Setup complete')

## Step 1: Deploy Interceptor Lambda

In [ ]:
import subprocess

result = subprocess.run(
    ['bash', 'deploy.sh'],
    cwd='gateway-setup/interceptor',
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print('❌ Error:', result.stderr)
else:
    print('\n✅ Interceptor Lambda deployed!')

## Step 2: Get Required ARNs

In [ ]:
# Get Interceptor Lambda ARN
try:
    interceptor_response = lambda_client.get_function(FunctionName='lakehouse-gateway-interceptor')
    INTERCEPTOR_ARN = interceptor_response['Configuration']['FunctionArn']
    print(f'✅ Interceptor ARN: {INTERCEPTOR_ARN}')
except Exception as e:
    print(f'❌ Error getting Interceptor ARN: {e}')
    INTERCEPTOR_ARN = None

# Get MCP Server Runtime ARN from SSM
MCP_SERVER_RUNTIME_ARN = ssm_client.get_parameter(Name='lh_mcp_server_runtime_arn')['Parameter']['Value']
print(f'✅ MCP Server ARN: {MCP_SERVER_RUNTIME_ARN}')

# Get Cognito User Pool ARN
COGNITO_USER_POOL_ID = ssm_client.get_parameter(Name='lh_cognito_user_pool_id')['Parameter']['Value']
COGNITO_USER_POOL_ARN = f'arn:aws:cognito-idp:{AWS_REGION}:{AWS_ACCOUNT_ID}:userpool/{COGNITO_USER_POOL_ID}'
print(f'✅ Cognito User Pool ARN: {COGNITO_USER_POOL_ARN}')

## Step 3: Create AgentCore Gateway

In [ ]:
result = subprocess.run([
    'python', 'create_gateway.py',
    '--gateway-name', 'lakehouse-gateway',
    '--mcp-server-runtime-arn', MCP_SERVER_RUNTIME_ARN,
    '--interceptor-arn', INTERCEPTOR_ARN,
    '--cognito-user-pool-arn', COGNITO_USER_POOL_ARN
], cwd='gateway-setup', capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print('❌ Error:', result.stderr)
else:
    print('\n✅ Gateway created!')
    print('\n📋 Save the Gateway ARN in the next step')

## Step 4: Save Gateway ARN to SSM

In [ ]:
# UPDATE THIS with actual ARN from Step 3
GATEWAY_ARN = 'arn:aws:bedrock-agentcore:region:account:gateway/id'  # CHANGE THIS
GATEWAY_ID = 'gateway-id'  # CHANGE THIS

ssm_client.put_parameter(Name='lh_gateway_arn', Value=GATEWAY_ARN, Type='String', Overwrite=True)
ssm_client.put_parameter(Name='lh_gateway_id', Value=GATEWAY_ID, Type='String', Overwrite=True)

print('✅ Saved Gateway configuration to SSM')

## Summary

✅ **Gateway & Interceptor Deployment Complete!**

**What was created:**
- Interceptor Lambda (JWT validation)
- AgentCore Gateway (routing)

**Next Steps:**
Run **06-deploy-agent.ipynb**